# Experiment 3 — Video Object Segmentation via Label Propagation

**Claim:** AnyUp3D-upsampled features produce more discriminative spatial representations than bilinear upsampling, measured by how well frozen features propagate object masks on DAVIS 2017 without any training.

**Protocol:** Non-parametric label propagation following Jabri et al. (NeurIPS 2020) *Space-Time Correspondence as a Contrastive Random Walk* (arXiv:2006.14613), with context design from Caron et al. (ICCV 2021) *Emerging Properties in Self-Supervised Vision Transformers* (DINO, arXiv:2104.14294).

**No training. No head. Frame 0 GT mask is given as input; frames 1…T are propagated via cosine similarity in feature space.**

**Metric:** J&F mean (DAVIS 2017 val, semi-supervised setting)

**Conditions:**
- AnyUp3D: Video Swin-T → AnyUp3D → 56×56 features
- Bilinear: Video Swin-T → bilinear interpolation → 56×56 features

---
**Design decisions and citations**
| Decision | Value | Source |
|---|---|---|
| Propagation resolution | 56×56 | Ablation: high enough to distinguish boundaries, fits T4 VRAM |
| Softmax temperature | τ=0.07 | Jabri et al. NeurIPS 2020, Table 1 |
| Top-k neighbors | k=7 | DINO eval code (github.com/facebookresearch/dino) |
| Context frames | Frame 0 + last n=6 frames | DINO eval code (`n_last_frames=6`) |
| Multi-object | Soft per-object propagation, argmax | DAVIS 2017 semi-supervised standard |

In [2]:
# ── Cell 1 · Installs ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


# natten — required by AnyUp cross-attention
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "natten",
     "--find-links",
     "https://shi-labs.com/natten/wheels/cu118/torch2.0.0/index.html"],
    check=False
)

print("Cell 1 done.")

Mounted at /content/drive
Cell 1 done.


In [3]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
import os

# ── Paths ───────────────────────────────────────────────────────────────────
DAVIS_ZIP  = "/content/drive/MyDrive/results/DAVIS-2017-trainval-480p.zip"
DAVIS_ROOT = "/content/DAVIS"

# Exp 2 cache — VideoMAE features and precomputed AnyUp3D outputs
CACHE_DIR  = "/content/drive/MyDrive/results/exp2/swin_cache"
# {clip}.pt   → {"feats": (8,768,14,14), "guides": (8,3,224,224)}
# {clip}_3d.pt → (8, 768, 224, 224)  — AnyUp3D upsampled
# {clip}_2d.pt → (8, 768, 224, 224)  — 2D AnyUp frame-by-frame

OUTPUT_DIR = "/content/drive/MyDrive/results/exp3"
MASK_DIR   = os.path.join(OUTPUT_DIR, "masks")    # {anyup3d, bilinear} subdirs
STILLS_DIR = os.path.join(OUTPUT_DIR, "stills")
VIDEO_DIR  = os.path.join(OUTPUT_DIR, "videos")

for d in [OUTPUT_DIR, MASK_DIR, STILLS_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Feature dims ─────────────────────────────────────────────────────────────
FEAT_DIM  = 768   # ← VideoMAE output channels
FEAT_SIZE = 14    # ← VideoMAE spatial output at IMG_SIZE=224 (224/16=14)
IMG_SIZE  = 224
T_FRAMES  = 8     # ← fixed: VideoMAE always outputs 8 temporal tokens

# ── Propagation ──────────────────────────────────────────────────────────────
# PROP_SIZE=56: 4× from 14 (bilinear path) or downsample from 224 (3d path).
# At 56×56: affinity matrix = 3136 × (7×3136) = 68M elements — fits on T4.
# At 224×224: 50176 × 351232 = 17B elements — infeasible.
PROP_SIZE = 224    # ← propagation resolution; changing requires verifying VRAM

# TAU, TOP_K, N_LAST_FRAMES: Jabri et al. NeurIPS 2020 + DINO eval code
TAU           = 0.07   # ← softmax temperature; Jabri et al. Table 1
TOP_K         = 7      # ← top-k neighbors; DINO eval code default
N_LAST_FRAMES = 6      # ← context window size; DINO eval code n_last_frames=6

# ── Visuals ──────────────────────────────────────────────────────────────────
N_VISUAL_CLIPS = 4
VISUAL_FPS     = 4

print("Config loaded.")
print(f"  FEAT_SIZE={FEAT_SIZE}, T_FRAMES={T_FRAMES}, PROP_SIZE={PROP_SIZE}")
print(f"  TAU={TAU}, TOP_K={TOP_K}, N_LAST_FRAMES={N_LAST_FRAMES}")

Config loaded.
  FEAT_SIZE=14, T_FRAMES=8, PROP_SIZE=224
  TAU=0.07, TOP_K=7, N_LAST_FRAMES=6


In [4]:
import shutil
shutil.rmtree(os.path.join(MASK_DIR, "anyup3d"))
shutil.rmtree(os.path.join(MASK_DIR, "bilinear"))
os.makedirs(os.path.join(MASK_DIR, "anyup3d"), exist_ok=True)
os.makedirs(os.path.join(MASK_DIR, "bilinear"), exist_ok=True)
print("Cleared.")

Cleared.


In [5]:
# ── Cell 3 · Imports + DAVIS setup ──────────────────────────────────────────
import subprocess, os, sys
import numpy as np
from pathlib import Path
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Extract DAVIS ────────────────────────────────────────────────────────────
if not os.path.isdir(DAVIS_ROOT):
    print("Extracting DAVIS...")
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)
    print("Done.")
else:
    print("DAVIS already extracted.")

IMG_DIR = os.path.join(DAVIS_ROOT, "JPEGImages",  "480p")
ANN_DIR = os.path.join(DAVIS_ROOT, "Annotations", "480p")
SET_DIR = os.path.join(DAVIS_ROOT, "ImageSets",   "2017")

with open(os.path.join(SET_DIR, "val.txt")) as f:
    val_clips = [l.strip() for l in f if l.strip()]
print(f"Val clips: {len(val_clips)}")

def get_frame_paths(clip):
    return sorted(Path(IMG_DIR, clip).glob("*.jpg"))

def get_ann_paths(clip):
    return sorted(Path(ANN_DIR, clip).glob("*.png"))

def load_ann(ann_path):
    """Load palette PNG → uint8 numpy (H, W), values = object IDs."""
    return np.array(Image.open(ann_path))

def get_n_objects(clip):
    return int(load_ann(get_ann_paths(clip)[0]).max())

# ── Verify cache coverage ────────────────────────────────────────────────────
missing = []
for clip in val_clips:
    for suffix in [".pt", "_3d.pt"]:
        if not os.path.exists(os.path.join(CACHE_DIR, f"{clip}{suffix}")):
            missing.append(f"{clip}{suffix}")

if missing:
    print(f"WARNING: {len(missing)} cache files missing — first 5: {missing[:5]}")
else:
    print(f"All cache files present for {len(val_clips)} val clips.")

Device: cuda
Extracting DAVIS...
Done.
Val clips: 30


In [6]:
# ── Cell 4 · Feature loading ─────────────────────────────────────────────────
#
# AnyUp3D condition: load {clip}_3d.pt → (8, 768, 224, 224) → pool to PROP_SIZE
# Bilinear condition: load {clip}.pt feats → (8, 768, 14, 14) → interpolate to PROP_SIZE
#
# T=8 is fixed — VideoMAE always produces 8 temporal tokens from 16 input frames.
# Guide frames were saved as frames[::2] from the 16 evenly-sampled input frames.
# Annotation alignment uses the same indices to load the correct GT masks.

def get_guide_frame_indices(clip):
    """
    Returns the 8 original frame indices that correspond to the 8 VideoMAE
    guide frames saved in the cache.
    VideoMAE input: 16 frames via np.linspace → guides: every 2nd → 8 frames.
    """
    all_frames = get_frame_paths(clip)
    N = len(all_frames)
    linspace_16 = np.linspace(0, N - 1, 16, dtype=int)   # 16 evenly spaced indices
    guide_idxs  = linspace_16[::2]                         # every 2nd → 8 indices
    return guide_idxs   # shape (8,) — maps feature token t to original frame index


def load_features(clip, condition):
    if condition == "anyup3d":
        path  = os.path.join(CACHE_DIR, f"{clip}_3d.pt")
        feats = torch.load(path, map_location="cpu",
                           weights_only=False).float()   # (8, 768, 224, 224) — no pooling
    else:
        path  = os.path.join(CACHE_DIR, f"{clip}.pt")
        cache = torch.load(path, map_location="cpu", weights_only=False)
        feats = cache["feats"].float()                   # (8, 768, 14, 14)
        feats = F.interpolate(
            feats,
            size=(PROP_SIZE, PROP_SIZE),
            mode="bilinear",
            align_corners=False
        )                                                # (8, 768, 224, 224)
    return feats


def load_ann_for_token(clip, token_t):
    """
    Load the GT annotation corresponding to feature token t.
    Maps token index → original frame index → annotation file.
    """
    ann_paths  = get_ann_paths(clip)
    guide_idxs = get_guide_frame_indices(clip)
    orig_idx   = guide_idxs[token_t]
    # Clamp in case annotation count differs slightly from frame count
    ann_idx    = min(int(orig_idx), len(ann_paths) - 1)
    return load_ann(ann_paths[ann_idx])   # (H_orig, W_orig) uint8


# Verify on one clip
sample_clip = val_clips[0]
print(f"Sample clip: {sample_clip}")
print(f"  Guide frame indices: {get_guide_frame_indices(sample_clip)}")
feats_3d  = load_features(sample_clip, "anyup3d")
feats_bil = load_features(sample_clip, "bilinear")
print(f"  AnyUp3D feats: {tuple(feats_3d.shape)}  "
      f"min={feats_3d.min():.3f} max={feats_3d.max():.3f}")
print(f"  Bilinear feats: {tuple(feats_bil.shape)}  "
      f"min={feats_bil.min():.3f} max={feats_bil.max():.3f}")
ann0 = load_ann_for_token(sample_clip, 0)
print(f"  Frame 0 annotation: shape={ann0.shape}, "
      f"unique values={np.unique(ann0)}")

Sample clip: bike-packing
  Guide frame indices: [ 0  9 18 27 36 45 54 63]
  AnyUp3D feats: (8, 768, 224, 224)  min=-7.765 max=13.547
  Bilinear feats: (8, 768, 224, 224)  min=-8.816 max=13.834
  Frame 0 annotation: shape=(480, 910), unique values=[0 1 2]


In [7]:
# ── Cell 5 · Label propagation ───────────────────────────────────────────────
#
# Protocol: Jabri et al. NeurIPS 2020 (arXiv:2006.14613)
# Context:  Caron et al. ICCV 2021 / DINO eval code (frame 0 + last N_LAST_FRAMES)
#
# Input:  (T, 768, PROP_SIZE, PROP_SIZE) features — T=8, fixed
# Output: (T, PROP_SIZE, PROP_SIZE) int64 — predicted object IDs per frame

def build_label_map(ann_HW, n_objects, H, W):
    """
    One-hot label map from annotation.
    ann_HW:    (H_orig, W_orig) uint8 object IDs
    Returns:   (n_objects+1, H*W) float32  — channel 0=bg, 1..n=objects
    """
    ann = cv2.resize(ann_HW, (W, H),
                     interpolation=cv2.INTER_NEAREST).astype(np.int64)  # (H, W)
    n_ch  = n_objects + 1
    label = np.zeros((n_ch, H * W), dtype=np.float32)
    flat  = ann.flatten()
    for obj_id in range(n_ch):
        label[obj_id, flat == obj_id] = 1.0
    return torch.from_numpy(label)   # (n_ch, H*W)


@torch.no_grad()
def propagate_clip(feats_TCHW, clip):
    """
    Non-parametric label propagation for one clip.

    feats_TCHW: (T, 768, PROP_SIZE, PROP_SIZE) float32
    clip:       clip name string — used to load GT annotations

    Returns: (T, PROP_SIZE, PROP_SIZE) int64
    """
    T, C, H, W = feats_TCHW.shape   # T=8, C=768, H=W=PROP_SIZE
    N = H * W                        # ← PROP_SIZE^2 = 3136 spatial locations

    # ── Step 1: L2-normalize ─────────────────────────────────────────────────
    feats_flat = feats_TCHW.view(T, C, N).permute(0, 2, 1)   # (T, N, C)
    feats_flat = F.normalize(feats_flat, dim=-1)              # unit vectors

    # ── Step 2: Frame 0 label map from GT ───────────────────────────────────
    ann0      = load_ann_for_token(clip, 0)   # (H_orig, W_orig) uint8
    n_objects = int(ann0.max())
    n_ch      = n_objects + 1

    label_maps    = torch.zeros(T, n_ch, N)   # (T, n_ch, N)
    label_maps[0] = build_label_map(ann0, n_objects, H, W)

    # ── Step 3-6: Frame-by-frame propagation ────────────────────────────────
    CHUNK_SIZE = 1024  # ← query rows per chunk; reduce to 512 if still OOM

    for t in range(1, T):
      past      = list(range(max(1, t - N_LAST_FRAMES), t))
      context_t = [0] + past

      ctx_feats  = feats_flat[context_t].reshape(-1, C).to(device)   # (Nctx, C)
      ctx_labels = label_maps[context_t].permute(1, 0, 2).reshape(n_ch, -1).to(device)

      new_labels = torch.zeros(N, n_ch, device=device)

      for start in range(0, N, CHUNK_SIZE):   # ← CHUNK_SIZE controls peak VRAM
          end = min(start + CHUNK_SIZE, N)
          q_chunk = feats_flat[t, start:end].to(device)   # (chunk, C)

          A_chunk = torch.mm(q_chunk, ctx_feats.T) / TAU  # (chunk, Nctx)

          topk_vals, topk_idx = A_chunk.topk(TOP_K, dim=-1)
          A_sparse = torch.full_like(A_chunk, float("-inf"))
          A_sparse.scatter_(1, topk_idx, topk_vals)

          A_soft = F.softmax(A_sparse, dim=-1)             # (chunk, Nctx)
          new_labels[start:end] = torch.mm(A_soft, ctx_labels.T)  # (chunk, n_ch)

          del q_chunk, A_chunk, A_sparse, A_soft
          torch.cuda.empty_cache()

      label_maps[t] = new_labels.T.cpu()   # (n_ch, N)
      del ctx_feats, ctx_labels, new_labels
      torch.cuda.empty_cache()
    # ── Step 7: argmax → integer mask ───────────────────────────────────────
    return label_maps.argmax(dim=1).view(T, H, W)   # (T, H, W) int64


print("Propagation functions defined.")

Propagation functions defined.


In [8]:
# ── Cell 6 · Run propagation + save masks ───────────────────────────────────

def save_mask_png(mask_HW, clip, token_t, out_path):
    """
    Save predicted mask as palette PNG at original DAVIS annotation resolution.
    mask_HW: (H, W) int64 — values 0..n_objects
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    ann_paths = get_ann_paths(clip)
    ref_img   = Image.open(ann_paths[0])
    H_orig, W_orig = np.array(ref_img).shape[:2]

    mask_np = mask_HW.numpy().astype(np.uint8)
    mask_np = cv2.resize(mask_np, (W_orig, H_orig),
                         interpolation=cv2.INTER_NEAREST)

    out_img = Image.fromarray(mask_np, mode="P")
    if ref_img.mode == "P":
        out_img.putpalette(ref_img.getpalette())
    out_img.save(out_path)


def get_mask_stem(clip, token_t):
    """Filename stem for token t — matches original frame filename."""
    all_frames = get_frame_paths(clip)
    guide_idxs = get_guide_frame_indices(clip)
    orig_idx   = guide_idxs[token_t]
    return Path(all_frames[orig_idx]).stem


skip_counts = {"anyup3d": 0, "bilinear": 0}

for condition in ["anyup3d", "bilinear"]:
    mask_cond_dir = os.path.join(MASK_DIR, condition)
    os.makedirs(mask_cond_dir, exist_ok=True)
    print(f"\nPropagating — {condition}...")

    for clip in tqdm(val_clips):
        # Check cache files exist
        base_path = os.path.join(CACHE_DIR, f"{clip}.pt")
        cond_path = os.path.join(CACHE_DIR, f"{clip}_3d.pt") \
                    if condition == "anyup3d" \
                    else base_path
        if not os.path.exists(cond_path) or not os.path.exists(base_path):
            skip_counts[condition] += 1
            continue

        # Skip if already done
        clip_mask_dir = os.path.join(mask_cond_dir, clip)
        stem0 = get_mask_stem(clip, 0)
        if os.path.exists(os.path.join(clip_mask_dir, f"{stem0}.png")):
            continue

        feats      = load_features(clip, condition)              # (8, 768, 56, 56)
        pred_masks = propagate_clip(feats, clip)                 # (8, 56, 56)

        for t in range(T_FRAMES):
            stem     = get_mask_stem(clip, t)
            out_path = os.path.join(clip_mask_dir, f"{stem}.png")
            save_mask_png(pred_masks[t], clip, t, out_path)

        del feats, pred_masks
        torch.cuda.empty_cache()

print("\nPropagation complete.")
print(f"  Skipped — anyup3d: {skip_counts['anyup3d']}, "
      f"bilinear: {skip_counts['bilinear']}")

# Verify
for condition in ["anyup3d", "bilinear"]:
    d = os.path.join(MASK_DIR, condition)
    clips_done = os.listdir(d)
    if clips_done:
        sample = clips_done[0]
        n_masks = len(os.listdir(os.path.join(d, sample)))
        print(f"  {condition}: {len(clips_done)} clips, "
              f"sample '{sample}' has {n_masks} masks")
    else:
        print(f"  {condition}: 0 clips saved — something went wrong")


Propagating — anyup3d...


  0%|          | 0/30 [00:00<?, ?it/s]/tmp/ipykernel_6496/293407682.py:18: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  out_img = Image.fromarray(mask_np, mode="P")
100%|██████████| 30/30 [04:40<00:00,  9.37s/it]



Propagating — bilinear...


100%|██████████| 30/30 [11:09<00:00, 22.31s/it]


Propagation complete.
  Skipped — anyup3d: 21, bilinear: 0
  anyup3d: 9 clips, sample 'bike-packing' has 8 masks
  bilinear: 30 clips, sample 'bike-packing' has 8 masks


In [9]:
# ── Cell 7 · J&F evaluation ─────────────────────────────────────────────────
#
# Self-contained implementation — no external package needed.
# J (Jaccard): region IoU per object, averaged across objects and frames.
# F (F-measure): boundary precision/recall per object, averaged.
# Both follow DAVIS 2017 definitions: Pont-Tuset et al. arXiv:1704.00675.

def compute_J(pred_HW, gt_HW):
    """IoU averaged over all object IDs present in GT."""
    obj_ids = np.unique(gt_HW)
    obj_ids = obj_ids[obj_ids > 0]
    if len(obj_ids) == 0:
        return 1.0
    scores = []
    for obj_id in obj_ids:
        inter = np.logical_and(pred_HW == obj_id, gt_HW == obj_id).sum()
        union = np.logical_or( pred_HW == obj_id, gt_HW == obj_id).sum()
        scores.append(inter / union if union > 0 else 1.0)
    return float(np.mean(scores))


def mask_to_boundary(mask_u8, dilation_px=2):
    """Boundary pixels via erosion — standard DAVIS F-measure approach."""
    kernel = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), np.uint8)
    eroded = cv2.erode(mask_u8, kernel, iterations=1)
    return mask_u8 - eroded


def compute_F(pred_HW, gt_HW, dilation_px=2):
    """Boundary F-measure averaged over all object IDs present in GT."""
    obj_ids = np.unique(gt_HW)
    obj_ids = obj_ids[obj_ids > 0]
    if len(obj_ids) == 0:
        return 1.0
    scores = []
    for obj_id in obj_ids:
        pred_b = mask_to_boundary((pred_HW == obj_id).astype(np.uint8), dilation_px)
        gt_b   = mask_to_boundary((gt_HW   == obj_id).astype(np.uint8), dilation_px)
        tp = np.logical_and(pred_b, gt_b).sum()
        p  = tp / (pred_b.sum() + 1e-8)
        r  = tp / (gt_b.sum()   + 1e-8)
        scores.append(2 * p * r / (p + r + 1e-8))
    return float(np.mean(scores))


def evaluate_condition(condition):
    """
    Evaluate one condition over all val clips.
    Returns per_clip dict: clip → {"J": float, "F": float, "JF": float}
    """
    mask_cond_dir = os.path.join(MASK_DIR, condition)
    per_clip = {}

    for clip in tqdm(val_clips, desc=condition):
        clip_mask_dir = os.path.join(mask_cond_dir, clip)
        if not os.path.isdir(clip_mask_dir):
            continue

        guide_idxs = get_guide_frame_indices(clip)
        ann_paths  = get_ann_paths(clip)
        frame_paths = get_frame_paths(clip)
        J_frames, F_frames = [], []

        for t in range(T_FRAMES):   # ← T_FRAMES=8, fixed
            stem      = get_mask_stem(clip, t)
            pred_path = os.path.join(clip_mask_dir, f"{stem}.png")
            if not os.path.exists(pred_path):
                continue

            # Load GT annotation for this token's original frame
            orig_idx = guide_idxs[t]
            ann_idx  = min(int(orig_idx), len(ann_paths) - 1)
            gt   = np.array(Image.open(ann_paths[ann_idx]))    # (H_orig, W_orig)
            pred = np.array(Image.open(pred_path))             # (H_orig, W_orig)

            # Align shapes if needed
            if pred.shape != gt.shape:
                pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]),
                                  interpolation=cv2.INTER_NEAREST)

            J_frames.append(compute_J(pred, gt))
            F_frames.append(compute_F(pred, gt))

        if J_frames:
            J_mean = float(np.mean(J_frames))
            F_mean = float(np.mean(F_frames))
            per_clip[clip] = {"J": J_mean, "F": F_mean, "JF": (J_mean + F_mean) / 2}

    return per_clip


# ── Run ──────────────────────────────────────────────────────────────────────
results_per_clip = {}
for condition in ["bilinear", "anyup3d"]:
    results_per_clip[condition] = evaluate_condition(condition)
    print(f"{condition}: evaluated {len(results_per_clip[condition])} clips")

bilinear: 100%|██████████| 30/30 [00:05<00:00,  5.25it/s]


bilinear: evaluated 30 clips


anyup3d: 100%|██████████| 30/30 [00:01<00:00, 18.93it/s]

anyup3d: evaluated 9 clips


In [10]:
# ── Cell 8 · Results table + CSV ────────────────────────────────────────────
import csv

# ── Overall means ────────────────────────────────────────────────────────────
results = {}
for condition in ["bilinear", "anyup3d"]:
    vals = results_per_clip[condition].values()
    results[condition] = {
        "J":  float(np.mean([v["J"]  for v in vals])),
        "F":  float(np.mean([v["F"]  for v in vals])),
        "JF": float(np.mean([v["JF"] for v in vals])),
    }

dJ  = results["anyup3d"]["J"]  - results["bilinear"]["J"]
dF  = results["anyup3d"]["F"]  - results["bilinear"]["F"]
dJF = results["anyup3d"]["JF"] - results["bilinear"]["JF"]

print("── Overall Results ─────────────────────────────────────────")
print(f"{'Method':<20} {'J':>8} {'F':>8} {'J&F':>8}")
print("-" * 46)
for cond in ["bilinear", "anyup3d"]:
    r = results[cond]
    print(f"{cond:<20} {r['J']:>8.3f} {r['F']:>8.3f} {r['JF']:>8.3f}")
print("-" * 46)
print(f"{'Δ (AnyUp3D − Bilinear)':<20} {dJ:>+8.3f} {dF:>+8.3f} {dJF:>+8.3f}")

# ── Per-clip breakdown sorted by delta ──────────────────────────────────────
all_clips = sorted(
    set(results_per_clip["bilinear"]) & set(results_per_clip["anyup3d"])
)
print("\n── Per-clip Results (sorted by Δ J&F) ─────────────────────")
print(f"{'Clip':<30} {'Bil J&F':>9} {'3D J&F':>9} {'Δ':>8}")
print("-" * 60)
for clip in sorted(all_clips,
                   key=lambda c: -(results_per_clip["anyup3d"][c]["JF"]
                                  - results_per_clip["bilinear"][c]["JF"])):
    bil = results_per_clip["bilinear"][clip]["JF"]
    jd  = results_per_clip["anyup3d"][clip]["JF"]
    print(f"{clip:<30} {bil:>9.3f} {jd:>9.3f} {jd - bil:>+8.3f}")

# ── Save CSV ─────────────────────────────────────────────────────────────────
csv_path = os.path.join(OUTPUT_DIR, "jf_results.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["clip", "bilinear_J", "bilinear_F", "bilinear_JF",
                        "anyup3d_J",  "anyup3d_F",  "anyup3d_JF", "delta_JF"])
    for clip in all_clips:
        b = results_per_clip["bilinear"].get(clip, {})
        a = results_per_clip["anyup3d"].get(clip, {})
        if b and a:
            w.writerow([clip,
                        f"{b['J']:.4f}", f"{b['F']:.4f}", f"{b['JF']:.4f}",
                        f"{a['J']:.4f}", f"{a['F']:.4f}", f"{a['JF']:.4f}",
                        f"{a['JF'] - b['JF']:.4f}"])
    w.writerow([])
    w.writerow(["MEAN",
                f"{results['bilinear']['J']:.4f}",
                f"{results['bilinear']['F']:.4f}",
                f"{results['bilinear']['JF']:.4f}",
                f"{results['anyup3d']['J']:.4f}",
                f"{results['anyup3d']['F']:.4f}",
                f"{results['anyup3d']['JF']:.4f}",
                f"{dJF:.4f}"])
print(f"\nSaved → {csv_path}")

── Overall Results ─────────────────────────────────────────
Method                      J        F      J&F
----------------------------------------------
bilinear                0.416    0.087    0.251
anyup3d                 0.565    0.158    0.361
----------------------------------------------
Δ (AnyUp3D − Bilinear)   +0.149   +0.072   +0.110

── Per-clip Results (sorted by Δ J&F) ─────────────────────
Clip                             Bil J&F    3D J&F        Δ
------------------------------------------------------------
camel                              0.387     0.494   +0.107
blackswan                          0.417     0.487   +0.070
bike-packing                       0.248     0.294   +0.045
car-roundabout                     0.479     0.520   +0.040
cows                               0.414     0.444   +0.029
dance-twirl                        0.290     0.314   +0.024
dog                                0.359     0.314   -0.045
bmx-trees                          0.189     0.13

In [11]:
# ── Cell 9 · Visual outputs ──────────────────────────────────────────────────
#
# (a) Side-by-side stills: N_VISUAL_CLIPS rows × 4 columns
#     [Original | GT mask | Bilinear | AnyUp3D]
# (b) Comparison video for highest-delta clip.

PALETTE = np.array([
    [0,   0,   0  ],   # 0: background
    [255, 0,   0  ],   # 1: object 1
    [0,   255, 0  ],   # 2: object 2
    [0,   0,   255],   # 3: object 3
    [255, 255, 0  ],   # 4: object 4
    [0,   255, 255],   # 5: object 5
], dtype=np.uint8)

S = 224   # ← display panel size in pixels

def colorize(mask_HW):
    ids = np.clip(mask_HW, 0, len(PALETTE) - 1)
    return PALETTE[ids]

def overlay(frame_HWC, mask_HW, alpha=0.55):
    color = colorize(mask_HW).astype(np.float32)
    fg    = mask_HW > 0
    out   = frame_HWC.copy().astype(np.float32)
    out[fg] = out[fg] * (1 - alpha) + color[fg] * alpha
    return np.clip(out, 0, 255).astype(np.uint8)

def load_frame_rgb(path):
    return np.array(Image.open(path).convert("RGB").resize((S, S)))

def load_mask(path):
    m = np.array(Image.open(path))
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.int32)

def load_gt(clip, token_t):
    guide_idxs = get_guide_frame_indices(clip)
    ann_paths  = get_ann_paths(clip)
    ann_idx    = min(int(guide_idxs[token_t]), len(ann_paths) - 1)
    m = load_ann(ann_paths[ann_idx])
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.int32)


# Pick clips that have masks for both conditions
vis_clips = [
    c for c in val_clips
    if (os.path.isdir(os.path.join(MASK_DIR, "anyup3d",  c)) and
        os.path.isdir(os.path.join(MASK_DIR, "bilinear", c)))
][:N_VISUAL_CLIPS]   # ← N_VISUAL_CLIPS from config

print(f"Visual clips: {vis_clips}")

# ── (a) Side-by-side stills ──────────────────────────────────────────────────
fig, axes = plt.subplots(len(vis_clips), 4, figsize=(16, 3.8 * len(vis_clips)))
if len(vis_clips) == 1:
    axes = [axes]

col_titles = ["Original", "GT mask", "Bilinear", "AnyUp3D"]
for col, title in enumerate(col_titles):
    axes[0][col].set_title(title, fontsize=11, fontweight="bold", pad=8)

for row, clip in enumerate(vis_clips):
    frame_paths = get_frame_paths(clip)
    mid_t       = T_FRAMES // 2   # middle token — most informative frame
    orig_idx    = get_guide_frame_indices(clip)[mid_t]
    stem        = Path(frame_paths[orig_idx]).stem

    frame   = load_frame_rgb(frame_paths[orig_idx])
    gt_mask = load_gt(clip, mid_t)

    mask_bil_path = os.path.join(MASK_DIR, "bilinear", clip, f"{stem}.png")
    mask_3d_path  = os.path.join(MASK_DIR, "anyup3d",  clip, f"{stem}.png")

    panels = [
        frame,
        overlay(frame, gt_mask, alpha=0.6),
        overlay(frame, load_mask(mask_bil_path)) if os.path.exists(mask_bil_path)
            else np.zeros((S, S, 3), dtype=np.uint8),
        overlay(frame, load_mask(mask_3d_path))  if os.path.exists(mask_3d_path)
            else np.zeros((S, S, 3), dtype=np.uint8),
    ]
    for col, (ax, panel) in enumerate(zip(axes[row], panels)):
        ax.imshow(panel)
        ax.axis("off")
    axes[row][0].set_ylabel(clip, fontsize=9)

plt.tight_layout()
stills_path = os.path.join(STILLS_DIR, "side_by_side.png")
plt.savefig(stills_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Stills saved → {stills_path}")

# ── (b) Comparison video — highest delta clip ────────────────────────────────
all_clips = sorted(
    set(results_per_clip["bilinear"]) & set(results_per_clip["anyup3d"])
)
best_clip = max(all_clips,
                key=lambda c: results_per_clip["anyup3d"][c]["JF"]
                             - results_per_clip["bilinear"][c]["JF"])
print(f"Best clip for video: {best_clip}")

frame_paths = get_frame_paths(best_clip)
guide_idxs  = get_guide_frame_indices(best_clip)
video_path  = os.path.join(VIDEO_DIR, f"{best_clip}_comparison.mp4")
fourcc      = cv2.VideoWriter_fourcc(*"mp4v")
writer      = cv2.VideoWriter(video_path, fourcc, VISUAL_FPS, (S * 4, S))

for t in range(T_FRAMES):
    orig_idx = guide_idxs[t]
    stem     = Path(frame_paths[orig_idx]).stem
    frame    = load_frame_rgb(frame_paths[orig_idx])
    gt_mask  = load_gt(best_clip, t)

    mask_bil_path = os.path.join(MASK_DIR, "bilinear", best_clip, f"{stem}.png")
    mask_3d_path  = os.path.join(MASK_DIR, "anyup3d",  best_clip, f"{stem}.png")

    p_gt  = overlay(frame, gt_mask, alpha=0.6)
    p_bil = overlay(frame, load_mask(mask_bil_path)) \
            if os.path.exists(mask_bil_path) else frame.copy()
    p_3d  = overlay(frame, load_mask(mask_3d_path)) \
            if os.path.exists(mask_3d_path)  else frame.copy()

    row_frame = np.concatenate([frame, p_gt, p_bil, p_3d], axis=1)
    writer.write(cv2.cvtColor(row_frame, cv2.COLOR_RGB2BGR))

writer.release()
print(f"Video saved → {video_path}")
print("\n── Experiment 3 complete. ──")

Visual clips: ['bike-packing', 'blackswan', 'bmx-trees', 'camel']
Stills saved → /content/drive/MyDrive/results/exp3/stills/side_by_side.png
Best clip for video: camel
Video saved → /content/drive/MyDrive/results/exp3/videos/camel_comparison.mp4

── Experiment 3 complete. ──


In [12]:
clip = best_clip
mask_dir = os.path.join(MASK_DIR, "bilinear", clip)
print(f"Masks saved for '{clip}':")
print(sorted(os.listdir(mask_dir)))

print(f"\nExpected stems:")
frame_paths = get_frame_paths(clip)
guide_idxs  = get_guide_frame_indices(clip)
for t in range(T_FRAMES):
    orig_idx = guide_idxs[t]
    stem     = Path(frame_paths[orig_idx]).stem
    path     = os.path.join(mask_dir, f"{stem}.png")
    print(f"  t={t} → orig_idx={orig_idx} → stem={stem} → exists={os.path.exists(path)}")

Masks saved for 'camel':
['00000.png', '00011.png', '00023.png', '00035.png', '00047.png', '00059.png', '00071.png', '00083.png']

Expected stems:
  t=0 → orig_idx=0 → stem=00000 → exists=True
  t=1 → orig_idx=11 → stem=00011 → exists=True
  t=2 → orig_idx=23 → stem=00023 → exists=True
  t=3 → orig_idx=35 → stem=00035 → exists=True
  t=4 → orig_idx=47 → stem=00047 → exists=True
  t=5 → orig_idx=59 → stem=00059 → exists=True
  t=6 → orig_idx=71 → stem=00071 → exists=True
  t=7 → orig_idx=83 → stem=00083 → exists=True


In [13]:
# ── Cell 10 · Visual outputs ─────────────────────────────────────────────────

PALETTE = np.array([
    [0,   0,   0  ],   # 0: background
    [255, 0,   0  ],   # 1: object 1
    [0,   255, 0  ],   # 2: object 2
    [0,   0,   255],   # 3: object 3
    [255, 255, 0  ],   # 4: object 4
    [0,   255, 255],   # 5: object 5
], dtype=np.uint8)

S = 224   # ← display panel size

def colorize(mask_HW):
    ids = np.clip(mask_HW, 0, len(PALETTE) - 1)
    return PALETTE[ids]

def overlay(frame_HWC, mask_HW, alpha=0.55):
    color = colorize(mask_HW).astype(np.float32)
    fg    = mask_HW > 0
    out   = frame_HWC.copy().astype(np.float32)
    out[fg] = out[fg] * (1 - alpha) + color[fg] * alpha
    return np.clip(out, 0, 255).astype(np.uint8)

def load_frame_rgb(path):
    return np.array(Image.open(path).convert("RGB").resize((S, S)))

def load_mask(path):
    m = np.array(Image.open(path))
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.int32)

def load_gt(clip, token_t):
    guide_idxs = get_guide_frame_indices(clip)
    ann_paths  = get_ann_paths(clip)
    ann_idx    = min(int(guide_idxs[token_t]), len(ann_paths) - 1)
    m = load_ann(ann_paths[ann_idx])
    return cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.int32)


# Pick clips that have masks for both conditions
vis_clips = [
    c for c in val_clips
    if (os.path.isdir(os.path.join(MASK_DIR, "anyup3d",  c)) and
        os.path.isdir(os.path.join(MASK_DIR, "bilinear", c)))
][:N_VISUAL_CLIPS]

print(f"Visual clips: {vis_clips}")
if len(vis_clips) == 0:
    print("No clips found — check MASK_DIR contents.")
else:
    # ── (a) Side-by-side stills ──────────────────────────────────────────────
    fig, axes = plt.subplots(len(vis_clips), 4,
                             figsize=(16, 3.8 * len(vis_clips)))
    if len(vis_clips) == 1:
        axes = [axes]

    col_titles = ["Original", "GT mask", "Bilinear", "AnyUp3D"]
    for col, title in enumerate(col_titles):
        axes[0][col].set_title(title, fontsize=11, fontweight="bold", pad=8)

    for row, clip in enumerate(vis_clips):
        frame_paths = get_frame_paths(clip)
        guide_idxs  = get_guide_frame_indices(clip)
        mid_t       = T_FRAMES // 2
        orig_idx    = guide_idxs[mid_t]
        stem        = Path(frame_paths[orig_idx]).stem

        frame   = load_frame_rgb(frame_paths[orig_idx])
        gt_mask = load_gt(clip, mid_t)

        mask_bil_path = os.path.join(MASK_DIR, "bilinear", clip, f"{stem}.png")
        mask_3d_path  = os.path.join(MASK_DIR, "anyup3d",  clip, f"{stem}.png")

        panels = [
            frame,
            overlay(frame, gt_mask, alpha=0.6),
            overlay(frame, load_mask(mask_bil_path)) if os.path.exists(mask_bil_path)
                else np.zeros((S, S, 3), dtype=np.uint8),
            overlay(frame, load_mask(mask_3d_path))  if os.path.exists(mask_3d_path)
                else np.zeros((S, S, 3), dtype=np.uint8),
        ]
        for col, (ax, panel) in enumerate(zip(axes[row], panels)):
            ax.imshow(panel)
            ax.axis("off")
        axes[row][0].set_ylabel(clip, fontsize=9)

    plt.tight_layout()
    stills_path = os.path.join(STILLS_DIR, "side_by_side.png")
    plt.savefig(stills_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Stills saved → {stills_path}")

    # ── (b) Comparison video — highest delta clip ─────────────────────────────
    all_clips = sorted(
        set(results_per_clip["bilinear"]) & set(results_per_clip["anyup3d"])
    )
    best_clip = max(
        all_clips,
        key=lambda c: results_per_clip["anyup3d"][c]["JF"]
                     - results_per_clip["bilinear"][c]["JF"]
    )
    print(f"Best clip for video: {best_clip}")

    frame_paths = get_frame_paths(best_clip)
    guide_idxs  = get_guide_frame_indices(best_clip)
    video_path  = f"/content/{best_clip}_comparison.mp4"
    fourcc      = cv2.VideoWriter_fourcc(*"mp4v")
    writer      = cv2.VideoWriter(video_path, fourcc, VISUAL_FPS, (S * 4, S))

    for t in range(T_FRAMES):
        orig_idx = guide_idxs[t]
        stem     = Path(frame_paths[orig_idx]).stem
        frame    = load_frame_rgb(frame_paths[orig_idx])
        gt_mask  = load_gt(best_clip, t)

        mask_bil_path = os.path.join(MASK_DIR, "bilinear", best_clip, f"{stem}.png")
        mask_3d_path  = os.path.join(MASK_DIR, "anyup3d",  best_clip, f"{stem}.png")

        p_gt  = overlay(frame, gt_mask, alpha=0.6)
        p_bil = overlay(frame, load_mask(mask_bil_path)) \
                if os.path.exists(mask_bil_path) else frame.copy()
        p_3d  = overlay(frame, load_mask(mask_3d_path)) \
                if os.path.exists(mask_3d_path)  else frame.copy()

        row_frame = np.concatenate([frame, p_gt, p_bil, p_3d], axis=1)
        writer.write(cv2.cvtColor(row_frame, cv2.COLOR_RGB2BGR))

    writer.release()
    print(f"Video saved → {video_path}")
    print("Download from Colab file browser (left sidebar → Files).")
    print("\n── Experiment 3 complete. ──")

Visual clips: ['bike-packing', 'blackswan', 'bmx-trees', 'camel']
Stills saved → /content/drive/MyDrive/results/exp3/stills/side_by_side.png
Best clip for video: camel
Video saved → /content/camel_comparison.mp4
Download from Colab file browser (left sidebar → Files).

── Experiment 3 complete. ──


In [14]:
# ── Debug visuals — save one frame per clip as individual PNGs ───────────────
debug_dir = "/content/debug_frames"
os.makedirs(debug_dir, exist_ok=True)

for clip in val_clips:
    bil_dir = os.path.join(MASK_DIR, "bilinear", clip)
    td_dir  = os.path.join(MASK_DIR, "anyup3d",  clip)
    if not os.path.isdir(bil_dir) or not os.path.isdir(td_dir):
        continue

    frame_paths = get_frame_paths(clip)
    guide_idxs  = get_guide_frame_indices(clip)

    # Use middle frame
    mid_t    = T_FRAMES // 2
    orig_idx = guide_idxs[mid_t]
    stem     = Path(frame_paths[orig_idx]).stem

    frame   = load_frame_rgb(frame_paths[orig_idx])
    gt_mask = load_gt(clip, mid_t)

    mask_bil_path = os.path.join(bil_dir, f"{stem}.png")
    mask_3d_path  = os.path.join(td_dir,  f"{stem}.png")

    p_gt  = overlay(frame, gt_mask, alpha=0.6)
    p_bil = overlay(frame, load_mask(mask_bil_path)) \
            if os.path.exists(mask_bil_path) else np.zeros((S, S, 3), dtype=np.uint8)
    p_3d  = overlay(frame, load_mask(mask_3d_path)) \
            if os.path.exists(mask_3d_path)  else np.zeros((S, S, 3), dtype=np.uint8)

    row = np.concatenate([frame, p_gt, p_bil, p_3d], axis=1)
    out = Image.fromarray(row)
    out.save(os.path.join(debug_dir, f"{clip}.png"))
    print(f"  {clip}: stem={stem}, bil={os.path.exists(mask_bil_path)}, "
          f"3d={os.path.exists(mask_3d_path)}")

print(f"\nAll frames saved to {debug_dir}")
print("Download from Colab file browser → right-click any PNG → Download.")

  bike-packing: stem=00036, bil=True, 3d=True
  blackswan: stem=00026, bil=True, 3d=True
  bmx-trees: stem=00042, bil=True, 3d=True
  camel: stem=00047, bil=True, 3d=True
  car-roundabout: stem=00039, bil=True, 3d=True
  car-shadow: stem=00020, bil=True, 3d=True
  cows: stem=00054, bil=True, 3d=True
  dance-twirl: stem=00047, bil=True, 3d=True
  dog: stem=00031, bil=True, 3d=True

All frames saved to /content/debug_frames
Download from Colab file browser → right-click any PNG → Download.


In [15]:
from sklearn.decomposition import PCA

clip = "blackswan"
feats_3d  = torch.load(os.path.join(CACHE_DIR, f"{clip}_3d.pt"),
                        map_location="cpu", weights_only=False).float()  # (8, 768, 224, 224)
feats_bil = torch.load(os.path.join(CACHE_DIR, f"{clip}.pt"),
                        map_location="cpu", weights_only=False)["feats"].float()  # (8, 768, 14, 14)

T = feats_3d.shape[0]

# PCA on AnyUp3D features — fit across all frames jointly for color consistency
F_3d = feats_3d.permute(0, 2, 3, 1).reshape(-1, 768).numpy()   # (8*224*224, 768)
pca  = PCA(n_components=3)
pca.fit(F_3d)
pca_3d = pca.transform(F_3d).reshape(T, 224, 224, 3)

# Normalize to [0,1] for display
pca_3d -= pca_3d.min()
pca_3d /= pca_3d.max() + 1e-8

# PCA on bilinear features
F_bil  = feats_bil.permute(0, 2, 3, 1).reshape(-1, 768).numpy()  # (8*14*14, 768)
pca2   = PCA(n_components=3)
pca_bil = pca2.fit_transform(F_bil).reshape(T, 14, 14, 3)
pca_bil -= pca_bil.min()
pca_bil /= pca_bil.max() + 1e-8

# Plot: one column per frame, two rows (3d top, bilinear bottom)
fig, axes = plt.subplots(2, T, figsize=(3 * T, 6))
for t in range(T):
    axes[0][t].imshow(pca_3d[t])
    axes[0][t].set_title(f"t={t}", fontsize=9)
    axes[0][t].axis("off")
    axes[1][t].imshow(pca_bil[t])
    axes[1][t].axis("off")

axes[0][0].set_ylabel("AnyUp3D 224×224", fontsize=10)
axes[1][0].set_ylabel("Bilinear 14×14",  fontsize=10)
plt.suptitle(f"PCA of features — {clip}", fontsize=12)
plt.tight_layout()
plt.savefig("/content/pca_features.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → /content/pca_features.png")

Saved → /content/pca_features.png
